In [2]:
# ============================================================
# TotalSegmentator 비교 테스트
# ------------------------------------------------------------
# 한 환자 기준:
#   A. 원본 spacing full volume에 TotalSegmentator 실행
#   B. z축 1mm resampled full volume에 TotalSegmentator 실행
#
# 비교:
#   - 둘 다 1mm CT grid로 맞춘 뒤
#   - organ별 Dice / IoU / volume 차이 / centroid 차이 계산
#   - 차이가 큰 slice overlay PNG 저장
#
# 주의:
#   정답 mask가 없으므로 "정확도"가 아니라
#   "두 전처리 조건 간 segmentation 일치도"를 보는 코드임.
# ============================================================

import os
import json
import shutil
import subprocess
import traceback
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import SimpleITK as sitk
from PIL import Image
from tqdm import tqdm


# ============================================================
# 1. CONFIG
# ============================================================

CONFIG = {
    # 다섯 명 테스트 대상
    "src_root": r"E:\jyp\ct_data_2d\Normal Cases",
    "num_random_patients": 5,
    "random_seed": 42,

    # 결과 저장 폴더
    "out_root": r"E:\jyp\totalseg_spacing_compare\normal001",

    # 방향 통일
    "orientation": "LPS",

    # z축 1mm resampling
    "target_z": 1.0,

    # TotalSegmentator 옵션
    # True면 -f 사용. 두 조건 모두 같은 옵션으로 돌려야 비교가 공정함.
    "use_fast_totalseg": True,

    # 기존 결과를 지우고 다시 돌릴지
    "overwrite_totalseg": False,

    # True면 아래 ROI만 실행. False면 전체 TotalSegmentator 실행.
    # 처음 테스트는 True 추천.
    "use_roi_subset": True,

    # 폐를 최종 폐 mask로 쓰려는 게 아니라,
    # 폐 주변 장기 위치 차이를 보려는 목적의 subset.
    # 만약 특정 class가 현재 TotalSegmentator 버전에서 안 맞으면 자동으로 full mode fallback 가능.
    "organ_roi_subset": [
        "heart",
        "aorta",
        "trachea",
        "esophagus",
        "liver",
        "stomach",
        "spleen",
        "pancreas",
        "vertebrae_T1",
        "vertebrae_T2",
        "vertebrae_T3",
        "vertebrae_T4",
        "vertebrae_T5",
        "vertebrae_T6",
        "vertebrae_T7",
        "vertebrae_T8",
        "vertebrae_T9",
        "vertebrae_T10",
        "vertebrae_T11",
        "vertebrae_T12",
    ],

    # ROI subset 실패 시 full segmentation으로 다시 시도할지
    "fallback_to_full_if_roi_fails": True,

    # PNG 확인용 window
    "hu_min": -1000,
    "hu_max": 400,

    # organ별로 차이 큰 slice 몇 장 저장할지
    "num_visual_slices_per_organ": 5,
}

args = SimpleNamespace(**CONFIG)

print("========== CONFIG ==========")
for k, v in CONFIG.items():
    print(f"{k}: {v}")


# ============================================================
# 2. 기본 함수
# ============================================================

def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def find_totalseg_executable():
    candidates = ["TotalSegmentator", "totalsegmentator"]

    for cmd in candidates:
        exe = shutil.which(cmd)
        if exe is not None:
            return cmd

    raise RuntimeError(
        "TotalSegmentator 실행 파일을 찾지 못했음. "
        "Jupyter 커널이 Python (lungprep_ts)인지 확인해야 함."
    )


def load_dicom_series_from_folder(series_dir: Path) -> sitk.Image:
    """
    환자 폴더 안 DICOM series 로드.
    series가 여러 개 있으면 slice 수가 가장 많은 series 사용.
    """

    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(str(series_dir))

    if not series_ids:
        raise RuntimeError(f"DICOM series를 찾지 못함: {series_dir}")

    best_series_id = None
    best_files = None
    best_count = -1

    for sid in series_ids:
        files = reader.GetGDCMSeriesFileNames(str(series_dir), sid)

        if len(files) > best_count:
            best_count = len(files)
            best_series_id = sid
            best_files = files

    if best_files is None or len(best_files) == 0:
        raise RuntimeError(f"DICOM files를 찾지 못함: {series_dir}")

    print(f"[DICOM] selected series id: {best_series_id}")
    print(f"[DICOM] number of slices: {len(best_files)}")

    reader.SetFileNames(best_files)
    img = reader.Execute()

    return img


def orient_image(img: sitk.Image, orientation: str) -> sitk.Image:
    return sitk.DICOMOrient(img, orientation)


def resample_z_only(
    img: sitk.Image,
    target_z: float,
    interpolator,
    default_value: float = 0.0,
) -> sitk.Image:
    """
    x/y spacing은 유지하고 z spacing만 target_z로 변경.
    CT에는 sitkLinear 사용.
    """

    original_spacing = img.GetSpacing()
    original_size = img.GetSize()

    new_spacing = (
        float(original_spacing[0]),
        float(original_spacing[1]),
        float(target_z),
    )

    new_size_z = int(round(original_size[2] * original_spacing[2] / target_z))
    new_size_z = max(new_size_z, 1)

    new_size = (
        int(original_size[0]),
        int(original_size[1]),
        int(new_size_z),
    )

    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetInterpolator(interpolator)
    resampler.SetDefaultPixelValue(default_value)

    return resampler.Execute(img)


def resample_to_reference(
    moving: sitk.Image,
    reference: sitk.Image,
    interpolator,
    default_value: float = 0.0,
) -> sitk.Image:
    """
    mask를 reference CT grid에 맞춤.
    mask는 반드시 nearest-neighbor 사용.
    """

    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference)
    resampler.SetInterpolator(interpolator)
    resampler.SetDefaultPixelValue(default_value)

    return resampler.Execute(moving)


def hu_to_uint8(slice_hu: np.ndarray, hu_min: int, hu_max: int) -> np.ndarray:
    x = np.clip(slice_hu.astype(np.float32), hu_min, hu_max)
    x = (x - hu_min) / float(hu_max - hu_min)
    x = np.clip(x * 255.0, 0, 255).astype(np.uint8)
    return x


# ============================================================
# 3. TotalSegmentator 실행 함수
# ============================================================

def run_totalsegmentator_for_compare(
    input_ct_path: Path,
    output_dir: Path,
    args,
    run_name: str,
):
    """
    TotalSegmentator 실행.
    ROI subset을 먼저 시도하고, 실패하면 선택적으로 full mode로 재시도.
    """

    if output_dir.exists() and not args.overwrite_totalseg:
        existing = list(output_dir.glob("*.nii.gz"))

        if len(existing) > 0:
            print(f"[TotalSegmentator:{run_name}] existing result reused: {output_dir}")
            return

    if output_dir.exists() and args.overwrite_totalseg:
        shutil.rmtree(output_dir)

    ensure_dir(output_dir)

    exe = find_totalseg_executable()

    base_cmd = [
        exe,
        "-i", str(input_ct_path),
        "-o", str(output_dir),
        "--nr_thr_resamp", "1",
        "--nr_thr_saving", "1",
    ]

    if args.use_fast_totalseg:
        base_cmd.append("-f")

    if args.use_roi_subset:
        cmd = base_cmd + ["--roi_subset"] + list(args.organ_roi_subset)
    else:
        cmd = base_cmd

    print(f"\n[TotalSegmentator:{run_name}] command")
    print(" ".join(cmd))

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    log_dir = Path(args.out_root) / "logs"
    ensure_dir(log_dir)

    stdout_path = log_dir / f"{run_name}_stdout.txt"
    stderr_path = log_dir / f"{run_name}_stderr.txt"

    stdout_path.write_text(result.stdout or "", encoding="utf-8", errors="replace")
    stderr_path.write_text(result.stderr or "", encoding="utf-8", errors="replace")

    print(f"[TotalSegmentator:{run_name}] returncode:", result.returncode)
    print("[stdout tail]")
    print((result.stdout or "")[-2000:])
    print("[stderr tail]")
    print((result.stderr or "")[-2000:])

    if result.returncode == 0:
        return

    print(f"[WARN] TotalSegmentator {run_name} ROI 실행 실패")

    if not args.fallback_to_full_if_roi_fails:
        raise RuntimeError(f"TotalSegmentator 실패: {run_name}")

    print(f"[TotalSegmentator:{run_name}] fallback full mode")

    if output_dir.exists():
        shutil.rmtree(output_dir)
    ensure_dir(output_dir)

    cmd_full = base_cmd

    print(" ".join(cmd_full))

    result_full = subprocess.run(
        cmd_full,
        capture_output=True,
        text=True,
    )

    stdout_full_path = log_dir / f"{run_name}_full_stdout.txt"
    stderr_full_path = log_dir / f"{run_name}_full_stderr.txt"

    stdout_full_path.write_text(result_full.stdout or "", encoding="utf-8", errors="replace")
    stderr_full_path.write_text(result_full.stderr or "", encoding="utf-8", errors="replace")

    print(f"[TotalSegmentator:{run_name}:full] returncode:", result_full.returncode)
    print("[stdout tail]")
    print((result_full.stdout or "")[-2000:])
    print("[stderr tail]")
    print((result_full.stderr or "")[-2000:])

    if result_full.returncode != 0:
        raise RuntimeError(f"TotalSegmentator full mode도 실패: {run_name}")


# ============================================================
# 4. mask 비교 함수
# ============================================================

def mask_to_bool_array(mask_img: sitk.Image) -> np.ndarray:
    arr = sitk.GetArrayFromImage(mask_img)
    return arr > 0


def dice_score(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool)
    b = b.astype(bool)

    inter = np.logical_and(a, b).sum()
    denom = a.sum() + b.sum()

    if denom == 0:
        return np.nan

    return float(2.0 * inter / denom)


def iou_score(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool)
    b = b.astype(bool)

    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    if union == 0:
        return np.nan

    return float(inter / union)


def centroid_mm(mask_arr_zyx: np.ndarray, reference_img: sitk.Image):
    """
    mask centroid를 mm 단위로 근사 계산.
    LPS 방향 통일 후 일반 spacing 기준으로 계산.
    """

    coords = np.argwhere(mask_arr_zyx > 0)

    if coords.shape[0] == 0:
        return (np.nan, np.nan, np.nan)

    # coords는 z,y,x 순서
    cz, cy, cx = coords.mean(axis=0)

    spacing = reference_img.GetSpacing()
    origin = reference_img.GetOrigin()

    # SimpleITK index는 x,y,z 순서
    x_mm = origin[0] + cx * spacing[0]
    y_mm = origin[1] + cy * spacing[1]
    z_mm = origin[2] + cz * spacing[2]

    return (float(x_mm), float(y_mm), float(z_mm))


def centroid_distance_mm(c1, c2):
    if any(np.isnan(v) for v in c1) or any(np.isnan(v) for v in c2):
        return np.nan

    return float(np.sqrt((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2 + (c1[2] - c2[2]) ** 2))


def volume_mm3(mask_arr: np.ndarray, reference_img: sitk.Image) -> float:
    spacing = reference_img.GetSpacing()
    voxel_volume = float(spacing[0] * spacing[1] * spacing[2])
    return float(mask_arr.sum() * voxel_volume)


def per_slice_metrics(native_arr, one_mm_arr, organ_name):
    """
    slice별 Dice와 area 차이 계산.
    """

    rows = []

    zdim = native_arr.shape[0]

    for z in range(zdim):
        a = native_arr[z]
        b = one_mm_arr[z]

        area_a = int(a.sum())
        area_b = int(b.sum())

        if area_a == 0 and area_b == 0:
            continue

        d = dice_score(a, b)
        iou = iou_score(a, b)

        rows.append({
            "organ_name": organ_name,
            "slice_index": int(z),
            "native_area_px": area_a,
            "one_mm_area_px": area_b,
            "abs_area_diff_px": int(abs(area_a - area_b)),
            "dice": d,
            "iou": iou,
        })

    return rows


def save_comparison_overlay(
    ct_arr: np.ndarray,
    native_mask: np.ndarray,
    one_mm_mask: np.ndarray,
    z: int,
    out_path: Path,
    hu_min: int,
    hu_max: int,
):
    """
    비교 overlay 저장.
    색 의미:
        native only = 초록
        1mm only = 빨강
        둘 다 겹침 = 노랑
    """

    ensure_dir(out_path.parent)

    img = hu_to_uint8(ct_arr[z], hu_min=hu_min, hu_max=hu_max)
    rgb = np.stack([img, img, img], axis=-1)

    native = native_mask[z].astype(bool)
    one_mm = one_mm_mask[z].astype(bool)

    overlap = native & one_mm
    native_only = native & (~one_mm)
    one_mm_only = one_mm & (~native)

    # red channel
    rgb[one_mm_only, 0] = 255
    rgb[one_mm_only, 1] = 40
    rgb[one_mm_only, 2] = 40

    # green channel
    rgb[native_only, 0] = 40
    rgb[native_only, 1] = 255
    rgb[native_only, 2] = 40

    # yellow overlap
    rgb[overlap, 0] = 255
    rgb[overlap, 1] = 255
    rgb[overlap, 2] = 40

    Image.fromarray(rgb.astype(np.uint8)).save(out_path)


# ============================================================
# 5. 5명 랜덤 비교 실험 실행부
# ------------------------------------------------------------
# 비교 목적:
#   1) mask 보간 자체의 손실
#   2) 1mm CT에서 TotalSegmentator를 새로 돌릴 때 생기는 차이
#   3) 최종 1mm grid에서 두 전략이 얼마나 다른지
# ============================================================

import random
from pathlib import Path

# ============================================================
# CONFIG 교체
# ============================================================

CONFIG_MULTI = {
    "src_root": r"E:\jyp\ct_data_2d\Normal Cases",
    "out_root": r"E:\jyp\totalseg_spacing_compare_5patients",

    "num_random_patients": 5,
    "random_seed": 42,

    "orientation": "LPS",
    "target_z": 1.0,

    "use_fast_totalseg": True,
    "overwrite_totalseg": False,

    "use_roi_subset": True,
    "organ_roi_subset": [
        "heart",
        "aorta",
        "trachea",
        "esophagus",
        "liver",
        "stomach",
        "spleen",
        "pancreas",
        "vertebrae_T1",
        "vertebrae_T2",
        "vertebrae_T3",
        "vertebrae_T4",
        "vertebrae_T5",
        "vertebrae_T6",
        "vertebrae_T7",
        "vertebrae_T8",
        "vertebrae_T9",
        "vertebrae_T10",
        "vertebrae_T11",
        "vertebrae_T12",
    ],

    "fallback_to_full_if_roi_fails": True,

    "hu_min": -1000,
    "hu_max": 400,

    # 이번 평균 실험에서는 overlay 저장을 줄임
    # 필요하면 True로 바꿔도 됨
    "save_overlay": False,
    "num_visual_slices_per_organ": 3,
}

args = SimpleNamespace(**CONFIG_MULTI)

print("========== CONFIG_MULTI ==========")
for k, v in CONFIG_MULTI.items():
    print(f"{k}: {v}")


# ============================================================
# 환자 5명 랜덤 선택
# ============================================================

def list_dicom_patient_dirs(src_root: Path):
    patient_dirs = sorted([p for p in src_root.iterdir() if p.is_dir()])

    valid = []

    for p in patient_dirs:
        dcm_count = len(list(p.rglob("*.dcm")))

        if dcm_count > 0:
            valid.append({
                "patient_id": p.name,
                "patient_dir": p,
                "dcm_count": dcm_count,
            })

    return valid


src_root = Path(args.src_root)
out_root = Path(args.out_root)
ensure_dir(out_root)

all_patients = list_dicom_patient_dirs(src_root)

print("\n========== Patient pool ==========")
print("valid patient count:", len(all_patients))

if len(all_patients) < args.num_random_patients:
    raise RuntimeError(
        f"환자 수 부족: 필요한 수 {args.num_random_patients}, 실제 {len(all_patients)}"
    )

random.seed(args.random_seed)
selected_patients = random.sample(all_patients, args.num_random_patients)

print("\n========== Selected patients ==========")
for p in selected_patients:
    print(p["patient_id"], "| DICOM:", p["dcm_count"])

with open(out_root / "compare_5patients_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG_MULTI, f, indent=2, ensure_ascii=False)

pd.DataFrame([
    {
        "patient_id": p["patient_id"],
        "patient_dir": str(p["patient_dir"]),
        "dcm_count": p["dcm_count"],
    }
    for p in selected_patients
]).to_csv(
    out_root / "selected_patients.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 추가 metric 함수
# ============================================================

def safe_abs_percent_diff(new_value, ref_value):
    if ref_value is None or np.isnan(ref_value) or abs(ref_value) < 1e-8:
        return np.nan

    return float(abs(new_value - ref_value) / abs(ref_value) * 100.0)


def compare_two_masks(mask_a, mask_b, reference_img, prefix):
    """
    두 mask 비교.
    반환:
        dice, iou, volume 차이, centroid 차이
    """

    d = dice_score(mask_a, mask_b)
    iou = iou_score(mask_a, mask_b)

    vol_a = volume_mm3(mask_a, reference_img)
    vol_b = volume_mm3(mask_b, reference_img)

    vol_abs_diff = float(abs(vol_b - vol_a))
    vol_abs_diff_percent = safe_abs_percent_diff(vol_b, vol_a)

    c_a = centroid_mm(mask_a, reference_img)
    c_b = centroid_mm(mask_b, reference_img)
    c_dist = centroid_distance_mm(c_a, c_b)

    return {
        f"{prefix}_dice": d,
        f"{prefix}_iou": iou,
        f"{prefix}_volume_a_mm3": vol_a,
        f"{prefix}_volume_b_mm3": vol_b,
        f"{prefix}_volume_abs_diff_mm3": vol_abs_diff,
        f"{prefix}_volume_abs_diff_percent": vol_abs_diff_percent,
        f"{prefix}_centroid_distance_mm": c_dist,
    }


# ============================================================
# 환자 1명 처리 함수
# ============================================================

def process_one_patient_spacing_strategy_compare(patient_info, args):
    patient_id = patient_info["patient_id"]
    patient_dir = Path(patient_info["patient_dir"])

    patient_out = Path(args.out_root) / patient_id
    ensure_dir(patient_out)

    dirs = {
        "ct_native": patient_out / "ct_native_lps",
        "ct_1mm": patient_out / "ct_1mm_lps",
        "totalseg_native": patient_out / "totalseg_native_spacing",
        "totalseg_1mm": patient_out / "totalseg_1mm_spacing",
        "native_to_1mm": patient_out / "native_masks_to_1mm",
        "native_to_1mm_back_native": patient_out / "native_masks_to_1mm_back_native",
        "one_mm_to_native": patient_out / "one_mm_masks_back_to_native",
        "diff_masks": patient_out / "diff_masks",
        "logs": patient_out / "logs",
    }

    for d in dirs.values():
        ensure_dir(d)

    print("\n" + "=" * 100)
    print(f"[Patient] {patient_id}")
    print(f"[Folder] {patient_dir}")

    # --------------------------------------------------------
    # 1. DICOM → native LPS → 1mm LPS
    # --------------------------------------------------------
    ct_raw = load_dicom_series_from_folder(patient_dir)
    ct_native = orient_image(ct_raw, args.orientation)

    ct_1mm = resample_z_only(
        img=ct_native,
        target_z=args.target_z,
        interpolator=sitk.sitkLinear,
        default_value=-1024,
    )

    print("[native]")
    print("size:", ct_native.GetSize())
    print("spacing:", ct_native.GetSpacing())

    print("[1mm]")
    print("size:", ct_1mm.GetSize())
    print("spacing:", ct_1mm.GetSpacing())

    ct_native_path = dirs["ct_native"] / f"{patient_id}_native_lps.nii.gz"
    ct_1mm_path = dirs["ct_1mm"] / f"{patient_id}_1mm_lps.nii.gz"

    sitk.WriteImage(ct_native, str(ct_native_path))
    sitk.WriteImage(ct_1mm, str(ct_1mm_path))

    # --------------------------------------------------------
    # 2. TotalSegmentator 2번 실행
    # --------------------------------------------------------
    # run_totalsegmentator_for_compare 함수가 args.out_root를 로그 폴더로 사용하므로
    # 환자별 out_root로 바꾼 임시 args를 사용
    patient_args = SimpleNamespace(**vars(args))
    patient_args.out_root = str(patient_out)

    run_totalsegmentator_for_compare(
        input_ct_path=ct_native_path,
        output_dir=dirs["totalseg_native"],
        args=patient_args,
        run_name=f"{patient_id}_native_spacing",
    )

    run_totalsegmentator_for_compare(
        input_ct_path=ct_1mm_path,
        output_dir=dirs["totalseg_1mm"],
        args=patient_args,
        run_name=f"{patient_id}_one_mm_spacing",
    )

    # --------------------------------------------------------
    # 3. 결과 mask 목록
    # --------------------------------------------------------
    native_files = sorted(dirs["totalseg_native"].glob("*.nii.gz"))
    one_mm_files = sorted(dirs["totalseg_1mm"].glob("*.nii.gz"))

    native_names = {p.name.replace(".nii.gz", ""): p for p in native_files}
    one_mm_names = {p.name.replace(".nii.gz", ""): p for p in one_mm_files}

    common_organs = sorted(set(native_names.keys()) & set(one_mm_names.keys()))

    print("native organ count:", len(native_names))
    print("1mm organ count:", len(one_mm_names))
    print("common organ count:", len(common_organs))

    if len(common_organs) == 0:
        raise RuntimeError(f"{patient_id}: 공통 organ mask가 없음")

    rows = []

    for organ_name in tqdm(common_organs, desc=f"Compare {patient_id} organs"):
        native_mask_img = sitk.ReadImage(str(native_names[organ_name]))
        one_mm_mask_img = sitk.ReadImage(str(one_mm_names[organ_name]))

        # ----------------------------------------------------
        # A_native
        # 원본 spacing에서 TotalSegmentator가 만든 mask
        # ----------------------------------------------------
        native_on_native_img = resample_to_reference(
            moving=native_mask_img,
            reference=ct_native,
            interpolator=sitk.sitkNearestNeighbor,
            default_value=0,
        )

        # ----------------------------------------------------
        # A_to_1mm
        # 원본 spacing mask를 1mm grid로 보간
        # 이게 "mask를 1mm로 보간해서 쓰는 전략"의 최종 mask
        # ----------------------------------------------------
        native_on_1mm_img = resample_to_reference(
            moving=native_mask_img,
            reference=ct_1mm,
            interpolator=sitk.sitkNearestNeighbor,
            default_value=0,
        )

        # ----------------------------------------------------
        # A_roundtrip
        # 원본 mask → 1mm → 원본 spacing
        # 원본 mask와 비교하면 mask 보간 자체 손실을 근사할 수 있음
        # ----------------------------------------------------
        native_1mm_back_native_img = resample_to_reference(
            moving=native_on_1mm_img,
            reference=ct_native,
            interpolator=sitk.sitkNearestNeighbor,
            default_value=0,
        )

        # ----------------------------------------------------
        # B_1mm
        # 1mm CT에서 TotalSegmentator가 직접 만든 mask
        # ----------------------------------------------------
        one_mm_on_1mm_img = resample_to_reference(
            moving=one_mm_mask_img,
            reference=ct_1mm,
            interpolator=sitk.sitkNearestNeighbor,
            default_value=0,
        )

        # ----------------------------------------------------
        # B_back_native
        # 1mm CT TotalSegmentator 결과를 원본 spacing으로 되돌림
        # 원본 mask와 비교하면 1mm 선행 segmentation 차이를 근사할 수 있음
        # ----------------------------------------------------
        one_mm_back_native_img = resample_to_reference(
            moving=one_mm_on_1mm_img,
            reference=ct_native,
            interpolator=sitk.sitkNearestNeighbor,
            default_value=0,
        )

        # 저장
        native_to_1mm_path = dirs["native_to_1mm"] / f"{organ_name}.nii.gz"
        native_roundtrip_path = dirs["native_to_1mm_back_native"] / f"{organ_name}.nii.gz"
        one_mm_back_path = dirs["one_mm_to_native"] / f"{organ_name}.nii.gz"

        sitk.WriteImage(native_on_1mm_img, str(native_to_1mm_path))
        sitk.WriteImage(native_1mm_back_native_img, str(native_roundtrip_path))
        sitk.WriteImage(one_mm_back_native_img, str(one_mm_back_path))

        # array 변환
        a_native = mask_to_bool_array(native_on_native_img)
        a_roundtrip = mask_to_bool_array(native_1mm_back_native_img)

        a_to_1mm = mask_to_bool_array(native_on_1mm_img)
        b_1mm = mask_to_bool_array(one_mm_on_1mm_img)
        b_back_native = mask_to_bool_array(one_mm_back_native_img)

        row = {
            "patient_id": patient_id,
            "organ_name": organ_name,

            "native_ct_size": str(ct_native.GetSize()),
            "native_ct_spacing": str(ct_native.GetSpacing()),
            "one_mm_ct_size": str(ct_1mm.GetSize()),
            "one_mm_ct_spacing": str(ct_1mm.GetSpacing()),

            "native_mask_nii": str(native_names[organ_name]),
            "one_mm_mask_nii": str(one_mm_names[organ_name]),
            "native_to_1mm_mask_nii": str(native_to_1mm_path),
            "native_roundtrip_mask_nii": str(native_roundtrip_path),
            "one_mm_back_native_mask_nii": str(one_mm_back_path),
        }

        # ----------------------------------------------------
        # 1) mask 보간 자체 손실
        # A_native vs A_roundtrip
        # ----------------------------------------------------
        row.update(compare_two_masks(
            mask_a=a_native,
            mask_b=a_roundtrip,
            reference_img=ct_native,
            prefix="mask_resample_roundtrip",
        ))

        # ----------------------------------------------------
        # 2) 1mm CT에서 TS 직접 실행 차이
        # A_native vs B_back_native
        # ----------------------------------------------------
        row.update(compare_two_masks(
            mask_a=a_native,
            mask_b=b_back_native,
            reference_img=ct_native,
            prefix="one_mm_ts_vs_native",
        ))

        # ----------------------------------------------------
        # 3) 최종 1mm grid에서 두 전략 차이
        # A_to_1mm vs B_1mm
        # ----------------------------------------------------
        row.update(compare_two_masks(
            mask_a=a_to_1mm,
            mask_b=b_1mm,
            reference_img=ct_1mm,
            prefix="strategy_gap_on_1mm",
        ))

        # error 크기 비교
        row["mask_resample_roundtrip_error_1_minus_dice"] = (
            1.0 - row["mask_resample_roundtrip_dice"]
            if not np.isnan(row["mask_resample_roundtrip_dice"]) else np.nan
        )

        row["one_mm_ts_vs_native_error_1_minus_dice"] = (
            1.0 - row["one_mm_ts_vs_native_dice"]
            if not np.isnan(row["one_mm_ts_vs_native_dice"]) else np.nan
        )

        row["strategy_gap_on_1mm_error_1_minus_dice"] = (
            1.0 - row["strategy_gap_on_1mm_dice"]
            if not np.isnan(row["strategy_gap_on_1mm_dice"]) else np.nan
        )

        if not np.isnan(row["mask_resample_roundtrip_error_1_minus_dice"]) and not np.isnan(row["one_mm_ts_vs_native_error_1_minus_dice"]):
            if row["one_mm_ts_vs_native_error_1_minus_dice"] > row["mask_resample_roundtrip_error_1_minus_dice"]:
                row["larger_error_source"] = "one_mm_total_segmentation_difference"
            elif row["one_mm_ts_vs_native_error_1_minus_dice"] < row["mask_resample_roundtrip_error_1_minus_dice"]:
                row["larger_error_source"] = "mask_resampling_roundtrip"
            else:
                row["larger_error_source"] = "same"
        else:
            row["larger_error_source"] = "unknown"

        rows.append(row)

    return rows


# ============================================================
# 5명 실행
# ============================================================

all_rows = []
error_rows = []

for p in selected_patients:
    try:
        rows = process_one_patient_spacing_strategy_compare(
            patient_info=p,
            args=args,
        )

        all_rows.extend(rows)

        pd.DataFrame(all_rows).to_csv(
            out_root / "all_patient_organ_compare_raw.csv",
            index=False,
            encoding="utf-8-sig",
        )

    except Exception as e:
        print("\n[ERROR]")
        print("patient:", p)
        print("error:", str(e))
        traceback.print_exc()

        error_rows.append({
            "patient_id": p["patient_id"],
            "patient_dir": str(p["patient_dir"]),
            "error": str(e),
        })

        pd.DataFrame(error_rows).to_csv(
            out_root / "compare_errors.csv",
            index=False,
            encoding="utf-8-sig",
        )


raw_df = pd.DataFrame(all_rows)

raw_path = out_root / "all_patient_organ_compare_raw.csv"
raw_df.to_csv(raw_path, index=False, encoding="utf-8-sig")

if len(error_rows) > 0:
    pd.DataFrame(error_rows).to_csv(
        out_root / "compare_errors.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("\n========== RAW RESULT ==========")
print("raw rows:", len(raw_df))
print("errors:", len(error_rows))
print("raw path:", raw_path)


# ============================================================
# organ별 평균
# ============================================================

metric_cols = [
    "mask_resample_roundtrip_dice",
    "mask_resample_roundtrip_error_1_minus_dice",
    "mask_resample_roundtrip_volume_abs_diff_percent",
    "mask_resample_roundtrip_centroid_distance_mm",

    "one_mm_ts_vs_native_dice",
    "one_mm_ts_vs_native_error_1_minus_dice",
    "one_mm_ts_vs_native_volume_abs_diff_percent",
    "one_mm_ts_vs_native_centroid_distance_mm",

    "strategy_gap_on_1mm_dice",
    "strategy_gap_on_1mm_error_1_minus_dice",
    "strategy_gap_on_1mm_volume_abs_diff_percent",
    "strategy_gap_on_1mm_centroid_distance_mm",
]

organ_mean_df = (
    raw_df
    .groupby("organ_name")[metric_cols]
    .agg(["mean", "std", "count"])
)

# multi-index 컬럼 정리
organ_mean_df.columns = [
    f"{a}_{b}" for a, b in organ_mean_df.columns
]

organ_mean_df = organ_mean_df.reset_index()

organ_mean_path = out_root / "organ_mean_compare_5patients.csv"
organ_mean_df.to_csv(organ_mean_path, index=False, encoding="utf-8-sig")


# ============================================================
# 전체 평균
# ============================================================

overall_rows = []

for col in metric_cols:
    overall_rows.append({
        "metric": col,
        "mean": float(raw_df[col].mean(skipna=True)),
        "std": float(raw_df[col].std(skipna=True)),
        "count": int(raw_df[col].count()),
    })

overall_df = pd.DataFrame(overall_rows)

overall_path = out_root / "overall_mean_compare_5patients.csv"
overall_df.to_csv(overall_path, index=False, encoding="utf-8-sig")


# ============================================================
# error source 비율
# ============================================================

source_count_df = (
    raw_df["larger_error_source"]
    .value_counts()
    .reset_index()
)

source_count_df.columns = ["larger_error_source", "count"]

source_count_path = out_root / "larger_error_source_count.csv"
source_count_df.to_csv(source_count_path, index=False, encoding="utf-8-sig")


print("\n========== SAVED ==========")
print("raw:", raw_path)
print("organ mean:", organ_mean_path)
print("overall mean:", overall_path)
print("larger source count:", source_count_path)

print("\n========== Larger error source count ==========")
display(source_count_df)

print("\n========== Overall mean ==========")
display(overall_df)

print("\n========== Organ mean sorted by strategy gap Dice ==========")
display(
    organ_mean_df.sort_values(
        "strategy_gap_on_1mm_dice_mean",
        ascending=True
    )
)

========== CONFIG ==========
src_root: E:\jyp\ct_data_2d\Normal Cases
num_random_patients: 5
random_seed: 42
out_root: E:\jyp\totalseg_spacing_compare\normal001
orientation: LPS
target_z: 1.0
use_fast_totalseg: True
overwrite_totalseg: False
use_roi_subset: True
organ_roi_subset: ['heart', 'aorta', 'trachea', 'esophagus', 'liver', 'stomach', 'spleen', 'pancreas', 'vertebrae_T1', 'vertebrae_T2', 'vertebrae_T3', 'vertebrae_T4', 'vertebrae_T5', 'vertebrae_T6', 'vertebrae_T7', 'vertebrae_T8', 'vertebrae_T9', 'vertebrae_T10', 'vertebrae_T11', 'vertebrae_T12']
fallback_to_full_if_roi_fails: True
hu_min: -1000
hu_max: 400
num_visual_slices_per_organ: 5
========== CONFIG_MULTI ==========
src_root: E:\jyp\ct_data_2d\Normal Cases
out_root: E:\jyp\totalseg_spacing_compare_5patients
num_random_patients: 5
random_seed: 42
orientation: LPS
target_z: 1.0
use_fast_totalseg: True
overwrite_totalseg: False
use_roi_subset: True
organ_roi_subset: ['heart', 'aorta', 'trachea', 'esophagus', 'liver', 'stomac

Compare normal015 organs: 100%|████████████████████████████████████████████████████████| 20/20 [01:00<00:00,  3.04s/it]



[Patient] normal004
[Folder] E:\jyp\ct_data_2d\Normal Cases\normal004
[DICOM] selected series id: 2ecdcb82eba14d4ab10bcf32c7ad2d9c
[DICOM] number of slices: 128
[native]
size: (512, 512, 128)
spacing: (0.6640625, 0.6640625, 2.0000000000000004)
m]
size: (512, 512, 256)
spacing: (0.6640625, 0.6640625, 1.0)

[TotalSegmentator:normal004_native_spacing] command
TotalSegmentator -i E:\jyp\totalseg_spacing_compare_5patients\normal004\ct_native_lps\normal004_native_lps.nii.gz -o E:\jyp\totalseg_spacing_compare_5patients\normal004\totalseg_native_spacing --nr_thr_resamp 1 --nr_thr_saving 1 -f --roi_subset heart aorta trachea esophagus liver stomach spleen pancreas vertebrae_T1 vertebrae_T2 vertebrae_T3 vertebrae_T4 vertebrae_T5 vertebrae_T6 vertebrae_T7 vertebrae_T8 vertebrae_T9 vertebrae_T10 vertebrae_T11 vertebrae_T12
[TotalSegmentator:normal004_native_spacing] returncode: 0
[stdout tail]

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Using 'fast' option: r

Compare normal004 organs: 100%|████████████████████████████████████████████████████████| 20/20 [00:53<00:00,  2.68s/it]



[Patient] normal036
[Folder] E:\jyp\ct_data_2d\Normal Cases\normal036
[DICOM] selected series id: 01afcf9039404635af11e89f4c6e7a1d
[DICOM] number of slices: 159
[native]
size: (512, 512, 159)
spacing: (0.673828125, 0.673828125, 2.0)
m]
size: (512, 512, 318)
spacing: (0.673828125, 0.673828125, 1.0)

[TotalSegmentator:normal036_native_spacing] command
TotalSegmentator -i E:\jyp\totalseg_spacing_compare_5patients\normal036\ct_native_lps\normal036_native_lps.nii.gz -o E:\jyp\totalseg_spacing_compare_5patients\normal036\totalseg_native_spacing --nr_thr_resamp 1 --nr_thr_saving 1 -f --roi_subset heart aorta trachea esophagus liver stomach spleen pancreas vertebrae_T1 vertebrae_T2 vertebrae_T3 vertebrae_T4 vertebrae_T5 vertebrae_T6 vertebrae_T7 vertebrae_T8 vertebrae_T9 vertebrae_T10 vertebrae_T11 vertebrae_T12
[TotalSegmentator:normal036_native_spacing] returncode: 0
[stdout tail]

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Using 'fast' option: resampli

...

,larger_error_source,count
0,one_mm_total_segmentation_difference,100



========== Overall mean ==========


,metric,mean,std,count
0,mask_resample_roundtrip_dice,1.000000,0.000000,100
1,mask_resample_roundtrip_error_1_minus_dice,0.000000,0.000000,100
2,mask_resample_roundtrip_volume_abs_diff_percent,0.000000,0.000000,100
3,mask_resample_roundtrip_centroid_distance_mm,0.000000,0.000000,100
4,one_mm_ts_vs_native_dice,0.940669,0.042515,100
5,one_mm_ts_vs_native_error_1_minus_dice,0.059331,0.042515,100
6,one_mm_ts_vs_native_volume_abs_diff_percent,2.054941,2.033635,100
7,one_mm_ts_vs_native_centroid_distance_mm,0.585100,0.568643,100
8,strategy_gap_on_1mm_dice,0.923327,0.043305,100
9,strategy_gap_on_1mm_error_1_minus_dice,0.076673,0.043305,100



========== Organ mean sorted by strategy gap Dice ==========


,organ_name,mask_resample_roundtrip_dice_mean,mask_resample_roundtrip_dice_std,mask_resample_roundtrip_dice_count,mask_resample_roundtrip_error_1_minus_dice_mean,mask_resample_roundtrip_error_1_minus_dice_std,mask_resample_roundtrip_error_1_minus_dice_count,mask_resample_roundtrip_volume_abs_diff_percent_mean,mask_resample_roundtrip_volume_abs_diff_percent_std,mask_resample_roundtrip_volume_abs_diff_percent_count,...,strategy_gap_on_1mm_dice_count,strategy_gap_on_1mm_error_1_minus_dice_mean,strategy_gap_on_1mm_error_1_minus_dice_std,strategy_gap_on_1mm_error_1_minus_dice_count,strategy_gap_on_1mm_volume_abs_diff_percent_mean,strategy_gap_on_1mm_volume_abs_diff_percent_std,strategy_gap_on_1mm_volume_abs_diff_percent_count,strategy_gap_on_1mm_centroid_distance_mm_mean,strategy_gap_on_1mm_centroid_distance_mm_std,strategy_gap_on_1mm_centroid_distance_mm_count
8,vertebrae_T1,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.144898,0.010503,5,3.489640,2.775941,5,0.595095,0.320522,5
12,vertebrae_T2,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.137435,0.015328,5,2.118861,1.415819,5,0.557878,0.280093,5
13,vertebrae_T3,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.129146,0.012955,5,2.137825,2.216518,5,0.643989,0.227455,5
16,vertebrae_T6,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.118439,0.024903,5,0.848374,0.854328,5,0.994683,0.255849,5
15,vertebrae_T5,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.118290,0.030362,5,1.920606,3.189411,5,0.927541,0.443106,5
14,vertebrae_T4,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.115949,0.014772,5,3.460754,1.840718,5,0.552457,0.326126,5
17,vertebrae_T7,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.104746,0.028175,5,3.523418,1.358841,5,0.788330,0.180673,5
18,vertebrae_T8,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.081967,0.013371,5,3.759213,1.509655,5,0.568539,0.136587,5
19,vertebrae_T9,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.080478,0.023846,5,2.229786,1.043059,5,0.636674,0.445507,5
7,trachea,1.0,0.0,5,0.0,0.0,5,0.0,0.0,5,...,5,0.073271,0.021576,5,1.777399,1.364493,5,0.952638,0.566653,5
